In [1]:
import requests
import pandas as pd
from calendar import monthrange
import os

In [2]:
import urllib3
from urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(InsecureRequestWarning)

In [3]:
headers = {
    'Cache-Control': 'no-cache',
    'Ocp-Apim-Subscription-Key': '647de712d0db43fd893d9b93655c4497',
}

def send_query(year, month, day, forecast=False):
    if forecast:
        api_url = f"https://apim.misoenergy.org/lgi/v1/forecast/{year}-{str(month).zfill(2)}-{str(day).zfill(2)}/load"
    else:
        api_url = f"https://apim.misoenergy.org/lgi/v1/real-time/{year}-{str(month).zfill(2)}-{str(day).zfill(2)}/demand/load-state-estimator"

    for _ in range(5):
        try:
            response = requests.get(api_url, headers=headers, verify=False)
            response.raise_for_status()
            break
        except:
            print(f"Got status code {response.status_code}. Retrying API call.")

    return response

In [5]:
df_list = []

for year in range(2022, 2025):
    print(f"Running year {year}...")
    for month in range(1, 13):
        if (year == 2022) and (month < 9):
            continue
        
        print(f"Running month {month}...") 
        end_day = monthrange(year, month)[1] + 1        
        for day in range(1, end_day):
            response = send_query(year, month, day, forecast=True)
            df = pd.DataFrame.from_records(response.json()['data'])
            df = (
                df.loc[df.localResourceZone != 'N/A']
                .copy()
                .reset_index(drop=True)
            )
            df_list.append(df)

response = send_query(2022, 8, 30, forecast=True)
df = pd.DataFrame.from_records(response.json()['data'])
df = (
    df.loc[df.localResourceZone != 'N/A']
    .copy()
    .reset_index(drop=True)
)
df_list.append(df)

In [14]:
df_list = []

for year in range(2015, 2025):
    print(f"Running year {year}...")
    for month in range(1, 13):
        print(f"Running month {month}...") 
        end_day = monthrange(year, month)[1] + 1        
        for day in range(1, end_day):
            response = send_query(year, month, day)
            df = pd.DataFrame.from_records(response.json()['data'])
            df = (
                df.loc[df.zone != 'N/A']
                .copy()
                .reset_index(drop=True)
            )
            df_list.append(df)

In [37]:
miso_load = pd.concat(df_list, ignore_index=True)
miso_load['end_hour'] = miso_load['timeInterval'].apply(lambda x: x['end'])
miso_load['timestamp'] = pd.to_datetime(miso_load['end_hour']) + pd.Timedelta(hours=5)
miso_load = (
    miso_load.rename(columns={'zone': 'subba', 'demand': 'value'})
    [['timestamp', 'subba', 'value']]
)

In [ ]:
miso_load.to_csv(f"../data/iso_load_profiles/miso.csv", index=False)

In [25]:
miso_zone_subba_map = {
    'Z1': '0001',
    'Z2': '0027',
    'Z3': '0035',
    'Z4': '0004',
    'Z5': '0035',
    'Z6': '0006',
    'Z7': '0027',
    'Z8': '8910',
    'Z9': '8910',
    'Z10': '8910'
}

miso_forecast = pd.concat(df_list, ignore_index=True)
miso_forecast['end_hour'] = miso_forecast['timeInterval'].apply(lambda x: x['end'])
miso_forecast['timestamp'] = pd.to_datetime(miso_forecast['end_hour']) + pd.Timedelta(hours=5)
miso_forecast['subba'] = miso_forecast['localResourceZone'].map(miso_zone_subba_map)
miso_forecast = (
    miso_forecast.groupby(['timestamp', 'subba'], as_index=False)
    ['loadForecast']
    .sum()
    .rename(columns={'loadForecast': 'value'})
)

In [29]:
miso_forecast.to_csv(f"../data/iso_load_profiles/miso_forecast.csv", index=False)